# Website Summarizer

## Project Summary

This project builds an AI-powered Website Summarizer that extracts content from a webpage and generates a concise, human-readable summary using OpenAI language models. The goal is to learn the core foundations of AI engineering, including API integration, prompt design, environment management, web content processing, and LLM-powered applications. This serves as a first practical introduction to building real-world AI systems.


In [10]:
import os                                            #lets program interact with the operating system. to access environment variables such as the API keys
from dotenv import load_dotenv                       #this reads the .env file and loads values from it into the python's environment
from scraper import fetch_website_contents           #imports my function from the scraper.py
from IPython.display import Markdown, display        #imports notebook display tools - renders markdown nicely
from openai import OpenAI                            #this gives the program the ability to send prompts to openAI models and recieve AI generated respones 

In [11]:
# load the .env file, get the openAI key and check for common mistakes before the notebook tries to call openAI 

load_dotenv(override=True)                #reads the files load values from .env | override is used to prevent python from usin a variable named openai_api_key incase it already exists on the system
api_key = os.getenv('OPENAI_API_KEY')     #Go into the environment and find the value stored under the name OPENAI_API_KEY

# Chceking weather the API key looks vaild

if not api_key:
    print("No API key was found")
elif not api_key.startswith("sk-proj-"):         #if it doesnt start with sk-
    print("An API key was found, but it doesn't start sk-proj-")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end")
else:
    print("API key found and looks good so far!")

API key found and looks good so far!


In [13]:
message = "Hello, GPT! this is a test!"

messages = [{"role": "user", "content": message}]

messages

[{'role': 'user', 'content': 'Hello, GPT! this is a test!'}]

In [ ]:
openai = OpenAI()                            #create an openai client object
response = openai.chat.completions.create(
    model="gpt-4o-mini", 
    messages=messages
    )
response.choices[0].message.content 

"Hello! I'm here and ready to assist you. How can I help you today?"

In [ ]:
# running my scraper function - which calls the function in imported from scraper.py

ed = fetch_website_contents("https://marvodyn.com")
print(ed)

marvodyn.com

Skip to content
Menu
Search
Home
Credit
Banking
Investing
Taxes
Budgeting & Saving
Loans
Students
Global Money
Tools
Resources
Shop
About
Close Menu
•
•
•
•
•
←
→
MARVODYN
Money Made Simple for Immigrants in America
Press Here to Start
BEGINNER GUIDES
March 6, 2026
Credit in America Explained for Immigrants: A Complete Beginner Guide
Credit
Why Credit Matters More Than You Might Think You made a big decision coming to the United States. You may…
March 6, 2026
Banking in America Explained for Immigrants: A Complete Beginner Guide
Banking
A System That Feels Unfamiliar at First When we arrive in the United States, one of the first practical challenges…
March 6, 2026
Taxes in America Explained for Immigrants: A Beginner’s Guide
Taxes
A System That Surprises Almost Every New Arrival When we come to the United States, taxes are rarely at the…
March 6, 2026
Financial Guide for International Students in the United States (Complete Beginner Guide)
Students
The Financial System No

In [ ]:
# Defining our system prompt - they tell the model how to behave

system_prompt = """
You are a helpful assistant with 20 years of experience that analyzes the contents of a website,
and provides a short summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""

In [ ]:
# Defining our user prompt - they tell the model what task to do with this particular data

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [ ]:
#function to build the messages list for the openai api  - prepare the exact data structure openai needs 
def messages_for(website):      #takes scraped website text and turns it into this ...
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

In [13]:

messages_for(ed)

[{'role': 'system',
  'content': '\nYou are a snarky assistant that analyzes the contents of a website,\nand provides a short, snarky, humorous summary, ignoring text that might be navigation related.\nRespond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nmarvodyn.com\n\nSkip to content\nMenu\nSearch\nHome\nCredit\nBanking\nInvesting\nTaxes\nBudgeting & Saving\nLoans\nStudents\nGlobal Money\nTools\nResources\nShop\nAbout\nClose Menu\n•\n•\n•\n•\n•\n←\n→\nMARVODYN\nMoney Made Simple for Immigrants in America\nPress Here to Start\nBEGINNER GUIDES\nMarch 6, 2026\nCredit in America Explained for Immigrants: A Complete Beginner Guide\nCredit\nWhy Credit Matters More Than You Might Think You made a big decision coming to the United States. You may…\nMarch 6, 2026\nBank

In [18]:
#my function that Takes the website URL, scrapes its text, sends that text to OpenAI, and returns the AI summary.
def summarize(url):
    website = fetch_website_contents(url)
    response = openai.chat.completions.create(
        model = "gpt-4.1-mini",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [ ]:
#calling the summarize function
summarize("https://marvodyn.com")

'# marvodyn.com: Money Made Simple for Immigrants in America\n\nWelcome to *Marvodyn*, the website that treats immigrants to the U.S. like financial newbies (because you kind of are) and then walks you through credit, banking, taxes, budgeting, and investing like a patient but no-nonsense friend. It’s all beginner-friendly, because let’s face it, American money stuff can be a confusing mess when you first arrive.\n\n### What’s New?  \n- Fresh beginner guides dropped on **March 6, 2026**, covering the essentials: Credit, Banking, Taxes, and even a special guide for international students who found out the financial system here was *not* on the syllabus.\n- Hot-off-the-press articles simplifying the best financial options for ITIN holders, no-SSN-needed bank accounts, no-fee checking accounts, and personal loans for immigrants who aren’t even on the credit radar yet.\n- Upcoming gem teased for **April 3, 2026**: “Best Dividend Stocks for…” something, because apparently the stock market a

In [16]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [ ]:
display_summary("https://marvodyn.com")          #all this does is go to my website, dowload the webpage HTML, remove the useless HTML elements, extract the readable text add teh text into a prompt and send it to openai, recieve the summary and display it 

# Marvodyn.com: Making Money Less Confusing for Immigrants

Welcome to Marvodyn, where the mysterious U.S. financial system is broken down into bite-sized, immigrant-friendly pieces of wisdom. Need to decode credit, banking, taxes, or investing without the usual jargon? They’ve got beginner guides fresh as of March 6, 2026, covering everything from how credit works to surviving taxes, even a special guide for international students who didn’t get a finance lesson with their study plans.

**Latest scoop:**  
- Fresh 2026 picks for the best banks and loans that don’t require a Social Security Number or even credit.  
- Top investment apps where you can start with just a buck (because apparently, money grows on apps).  
- Best ways to send money back home without losing your shirt.

Newsflash: They dropped a brand new *Best Dividend Stocks* article on April 3, 2026 — because who doesn’t want their money to work harder than they do?

In summary: if you’re an immigrant trying not to drown in U.S. financial mumbo jumbo, Marvodyn is like that savvy friend who’s been there, done that, and now wants you to cash in without the headaches.